# optimizer-repr-string — worked example 1: Adam __repr__ with three hyperparameters

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-repr-string`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A `__repr__` method on a custom optimizer returns a human-readable string summarizing its hyperparameters — useful in log files and tqdm postfix displays. The conventional format is `ClassName(hparam1=value1, hparam2=value2, ...)`. Crucially, the large parameter tensors stored in `self.params` are NOT included, only the scalar hyperparameters.

## Worked solution

**Step 1 — Identify the hyperparameters to display.**
For our Adam-like optimizer, the relevant hyperparameters are `lr`, `betas`, and `eps`. We exclude `self.params` because it is a large list of tensors — printing it would produce pages of output.

**Step 2 — Use an f-string.**
The `__repr__` method returns a single f-string. We interpolate each hyperparameter attribute directly: `{self.lr}`, `{self.betas}`, `{self.eps}`. Python's default `repr` for floats and tuples is readable and precise.

**Step 3 — Verify the format.**
We check that `repr(opt)` matches the expected string exactly, and that `str(opt)` returns the same string (since `__str__` defaults to `__repr__` when not defined).

**Step 4 — Confirm params are excluded.**
We check that the repr string does NOT contain any representation of tensor data.

In [ ]:
import torch as t

class AdamLike:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.betas  = betas
        self.eps    = eps

    def __repr__(self):
        return f'AdamLike(lr={self.lr}, betas={self.betas}, eps={self.eps})'

# --- exercise it ---
t.manual_seed(0)
params = [t.tensor([1.0, 2.0], requires_grad=True),
          t.tensor([3.0],       requires_grad=True)]

opt = AdamLike(params, lr=3e-4, betas=(0.9, 0.999), eps=1e-8)
rep = repr(opt)
print(rep)
assert rep == 'AdamLike(lr=0.0003, betas=(0.9, 0.999), eps=1e-08)'
assert str(opt) == repr(opt), '__str__ should fall back to __repr__'
assert 'tensor' not in rep.lower(), 'repr should not contain tensor data'
print('repr format correct!')